<img src="https://raw.githubusercontent.com/Nextia-Intelligence/nextia-learning-labs/main/nextia-notebook-branding/assets/courses/c34-header.png" alt="Nextia Learning · Anomaly Detection: Find What Is Unusual" width="100%">

# Solution: Where an LLM helps, and where it does not

**[Where an LLM helps, and where it does not](https://learning.nextia-ai.com/courses/anomaly-detection/m06/where-an-llm-helps/)** · Anomaly Detection: Find What Is Unusual · Module 6, lesson 3 of 4 · [learning.nextia-ai.com](https://learning.nextia-ai.com)

**You will learn to** read recorded summaries that a language model wrote for 20 alert groups, check every summary against its facts with code, and see what such a check finds and what it misses.

| | |
|---|---|
| **Time** | About 30 minutes |
| **Needs** | An internet connection for the setup cells (pandas and NumPy, and a 22 kB file). No account, no key, no model call: the answers were recorded once. |
| **You should know** | The alert budget and alert groups, from [The payment that did not fit](https://learning.nextia-ai.com/courses/anomaly-detection/m01/the-payment-that-did-not-fit/). The lesson of this notebook explains what an LLM is. |
| **You do not need** | An API key or a paid service. |
| **Tested** | Python 3.12.13 in Jupyter on macOS (Apple silicon), from a fresh start (*Restart and run all*), with the package versions of the setup cell |

### How to use this notebook

This notebook has every query, task and check of the lesson. The [lesson page](https://learning.nextia-ai.com/courses/anomaly-detection/m06/where-an-llm-helps/) has the full explanations, the diagrams and the knowledge checks that count toward your certificate. Keep both open.

1. Run the cells in order, from the top. Press **Shift+Enter** to run a cell and move to the next one.
2. At a **Predict** note, write your prediction down before you run the cell.
3. At a **Your turn** note, write your answer in the cell, run it, then run the check cell below it.
4. If you are stuck, open the [solution notebook](https://github.com/Nextia-Intelligence/nextia-learning-labs/blob/main/C34/notebooks/M06-L03-where-an-llm-helps/llm-summaries-solution.ipynb).

## Setup: install the packages

Run the next cell. It installs the versions that the course uses: pandas 3.1.1, numpy 2.5.4, scipy 1.18.1, scikit-learn 1.9.1, matplotlib 3.11.2. It needs an internet connection. The first time, it takes up to a minute.

You should see one line per package, with the same versions. If you see other versions, restart the kernel (in Colab: **Runtime › Restart session**; in Kaggle: **Run › Restart & clear cell outputs**), then run this cell again. A restart is needed when an older version was already loaded.

In [ ]:
%pip install -q --disable-pip-version-check pandas==3.1.1 numpy==2.5.4 scipy==1.18.1 scikit-learn==1.9.1 matplotlib==3.11.2
from importlib.metadata import version
for p in ['pandas', 'numpy', 'scipy', 'scikit-learn', 'matplotlib']:
    print(p, version(p))

## Setup: get the recorded summaries

Run the next cell. It downloads the file `llm-alert-summaries.jsonl` into a folder `anomaly-course/recordings`, and checks its checksum. It needs an internet connection and no account.

You should see `Ready:` and the file with its size. If you see `Could not download`, check your internet connection (in Kaggle, turn on **Internet** in the notebook settings). If you see `wrong checksum`, delete the folder `anomaly-course`, then run the cell again.

The payments behind these summaries are made up for this course: no real people, cards or payments.

In [ ]:
# Setup: download the recorded summaries into anomaly-course/recordings/ and check the file.
# Standard library only.
import hashlib, os, shutil, subprocess, urllib.request
from pathlib import Path

URL = "https://raw.githubusercontent.com/Nextia-Intelligence/nextia-learning-labs/main/C34/recordings/llm-alert-summaries.jsonl"
EXPECTED = "65e7f1d89641e955cca374adc31a32d3e6211adf71ea21a92aec8cb3c781fc19"
HERE = Path.cwd()
if Path(".nextia-notebook").exists():          # the cell ran before: start again from the parent folder
    os.chdir("..")
    HERE = Path.cwd()
WORK = Path("anomaly-course").resolve()
(WORK / "recordings").mkdir(parents=True, exist_ok=True)
(WORK / ".nextia-notebook").touch()
path = WORK / "recordings" / "llm-alert-summaries.jsonl"

def sha256(p):
    return hashlib.sha256(p.read_bytes()).hexdigest()

if not (path.exists() and sha256(path) == EXPECTED):
    if (HERE / path.name).exists():            # a copy next to the notebook
        shutil.copy(HERE / path.name, path)
    else:
        try:
            urllib.request.urlretrieve(URL, path)
        except OSError:                        # Python without certificates (python.org on macOS): try curl
            if subprocess.run(["curl", "-fsSL", "-o", str(path), URL]).returncode:
                raise SystemExit(f"Could not download {URL}. Check your internet connection.")
if sha256(path) != EXPECTED:
    raise SystemExit("llm-alert-summaries.jsonl has the wrong checksum. Delete the folder anomaly-course and run again.")
os.chdir(WORK)
print("Ready:", path.name, f"({path.stat().st_size / 1e3:.1f} kB)")

## Setup: helpers

The next cell defines the check helpers `expect` and `expect_hash`. They tell you if your answer is right, without showing it. It also loads pandas as `pd` and NumPy as `np`. Run the cell. It prints nothing.

In [ ]:
# Helpers: check an answer without showing it, and the chart style.
import hashlib, json, warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
pd.set_option("display.precision", 3)
pd.set_option("display.width", 140)
pd.set_option("display.max_columns", 20)
warnings.filterwarnings("ignore", category=FutureWarning)
plt.rcParams.update({"figure.figsize": (9, 3.2), "axes.spines.top": False, "axes.spines.right": False})

def fingerprint(value):
    def norm(v):
        if hasattr(v, "to_dict") and hasattr(v, "columns"):
            return {"columns": [str(c) for c in v.columns], "rows": [norm(list(r)) for r in v.itertuples(index=False)]}
        if hasattr(v, "tolist"):
            return norm(v.tolist())
        if isinstance(v, dict):
            return {str(k): norm(x) for k, x in sorted(v.items(), key=lambda kv: str(kv[0]))}
        if isinstance(v, (list, tuple, set)):
            return [norm(x) for x in (sorted(v, key=str) if isinstance(v, set) else v)]
        if isinstance(v, bool) or v is None or isinstance(v, int):
            return v
        if isinstance(v, float):
            return None if v != v else round(v, 4)
        if hasattr(v, "item"):
            return norm(v.item())
        return str(v)
    return hashlib.sha256(json.dumps(norm(value), sort_keys=True).encode()).hexdigest()[:16]

def expect(what, yours, expected, tolerance=1e-6):
    """Compare one number or value with the expected one."""
    if isinstance(yours, (np.integer, np.floating)):
        yours = yours.item()
    if isinstance(expected, float) and isinstance(yours, (int, float)):
        same = abs(yours - expected) <= tolerance * max(1, abs(expected))
    else:
        same = yours == expected
    if same:
        print(f"Check passed: {what} is {yours}.")
    elif isinstance(expected, (int, float)) and isinstance(yours, (int, float)):
        print(f"Not yet: {what} is {yours}, which is too {'high' if yours > expected else 'low'}.")
    else:
        print(f"Not yet: {what} is {yours!r}, which is not the expected value.")

def expect_hash(what, yours, expected):
    """Compare any result with the fingerprint of the expected one."""
    if yours is None or yours is Ellipsis:
        print(f"Not yet: {what} has no value. Write your answer in the cell above, then run it again.")
    elif fingerprint(yours) == expected:
        print(f"Check passed: {what} is right.")
    else:
        print(f"Not yet: {what} is not the expected result. Read the task again, and check each step.")

<img src="https://raw.githubusercontent.com/Nextia-Intelligence/nextia-learning-labs/main/nextia-notebook-branding/assets/section-divider-light.png" alt="" width="100%">

## 1. One alert group and its summary

The detector (Isolation Forest on the behaviour features) chose its 50 alerts of each day. The alerts of one customer on one day form an **alert group**. For 20 groups, a small hosted language model got a short table of **facts** and wrote a summary and one "First check" for Sofia's team. Each line of the file is one group: the facts, the summary, and the model's tokens, cost and time.

Run the next cell. It shows the first group.

In [ ]:
import json, re
records = [json.loads(line) for line in open("recordings/llm-alert-summaries.jsonl", encoding="utf-8")]
print(len(records), "recorded summaries, model", records[0]["model"], "through the deployment", records[0]["deployment"])
first = records[0]
print(json.dumps(first["facts"], indent=1))
print()
print(first["summary"])

The model did not read the payments table. It got about 20 facts that the code had already computed, and turned them into a few sentences.

Run the next cell to see what the 20 calls cost.

> **Check.** You should see a mean of about 325 tokens in and 166 tokens out, and a total of US$0.0023.

In [ ]:
usage = pd.DataFrame({"tokens in": [r["tokens_in"] for r in records], "tokens out": [r["tokens_out"] for r in records],
                      "US$": [r["usd"] for r in records], "seconds": [r["latency_s"] for r in records]})
print(usage.agg(["mean", "median", "max"]).round(6).to_string())
print("Total for the 20 calls: US$", round(usage["US$"].sum(), 4))

## 2. Check every summary against its facts

A model can write a number that is not in its facts. Sofia must not have to check each summary by hand, so code checks it. The rule is simple: every number and every two-letter country code in the summary must be in the facts. The summary must have a "First check:" line, and it must not say that the payment is fraud.

> **Check.** You should see 0 summaries with a problem, of 20.

In [ ]:
def numbers(text):
    """Every number in a text: 1.03, 4,658.72, 13 ..."""
    return [float(x.replace(",", "")) for x in re.findall(r"(?<![\w-])\d[\d,]*(?:\.\d+)?", text)]

def check(record, date_parts=True):
    """The problems of one summary: a number or a country that is not in its facts, no 'First check:' line,
    or the words 'is fraud' (the reviewer decides, not the model)."""
    facts = record["facts"]
    allowed = set()
    for value in facts.values():
        for x in (value if isinstance(value, list) else [value]):
            if isinstance(x, (int, float)) and not isinstance(x, bool):
                allowed |= {round(float(x), 2), float(round(x)), float(int(x))}
            elif isinstance(x, str):
                allowed |= set(numbers(x))
    allowed |= {float(len(facts["amounts_cad"])), 1.0}
    if date_parts:            # "September 25" is the day of the facts; "16:27" is a time of the facts
        allowed |= {float(x) for x in facts["day"].split("-")}
        for t in (facts["first_payment_time"], facts["last_payment_time"]):
            allowed |= {float(x) for x in t.split(":")}
    summary = record["summary"]
    problems = [f"number {n:g} is not in the facts" for n in numbers(summary) if round(n, 2) not in allowed]
    for code in re.findall(r"\b[A-Z]{2}\b", summary):
        if code not in facts["countries"] and code not in ("OK", "ID"):
            problems.append(f"country {code} is not in the facts")
    if "first check:" not in summary.lower():
        problems.append("no 'First check:' line")
    if re.search(r"\b(is|was|confirmed|clearly) fraud", summary.lower()):
        problems.append("says it is fraud")
    return problems

results = pd.DataFrame({"customer": [r["facts"]["customer"] for r in records], "day": [r["facts"]["day"] for r in records],
                        "problems": [check(r) for r in records]})
print("Summaries with a problem:", sum(len(p) > 0 for p in results.problems), "of", len(results))

Zero problems is good news only if the check can find a problem. So test the check: change one number on purpose and run it again.

> **Check.** You should see one problem: number 10.3 is not in the facts.

In [ ]:
broken = json.loads(json.dumps(first))                      # a copy, then one wrong number on purpose
broken["summary"] = broken["summary"].replace("1.03", "10.30")
print(broken["summary"].splitlines()[0])
print(check(broken))

### A check that is too strict

The first version of this check did not know that a summary may write the date in words, for example "September 25". It took "25" for an unknown number. Run the next cell: it uses that first version (`date_parts=False`).

> **Check.** You should see 3 summaries flagged, and each one wrote the date in words.

In [ ]:
for r in records:
    strict = [p for p in check(r, date_parts=False) if p.startswith("number")]
    if strict:
        print(r["facts"]["customer"], r["facts"]["day"], strict, "|", r["summary"].splitlines()[0][:80])

A check that is too strict sends false alarms about the summaries. Then people stop reading its warnings, exactly as with a detector that sends too many alerts.

## 3. Check yourself

**Your turn 1.** The number check cannot see a wrong **comparison**. One summary says that all the amounts were above the customer's usual amount, but one amount was not. Find it: look for "each was above", "all were above" or "the amounts were above" in the summary, and compare `amounts_cad` with `customer_usual_amount_cad`. Store the customer ID as `wrong_claim`.

In [ ]:
wrong_claim = None
for r in records:
    f = r["facts"]
    says_all_above = re.search(r"\b(each|all|the amounts) (was|were) above", r["summary"])
    usual = f["customer_usual_amount_cad"]          # None for a customer with no earlier payments
    if says_all_above and usual is not None and any(a <= usual for a in f["amounts_cad"]):
        wrong_claim = f["customer"]
print(wrong_claim)

In [ ]:
expect_hash('the customer of the wrong comparison', wrong_claim, '5df484ae2101043d')

**Your turn 2.** Tomás wants the model to read **every** payment instead of the detector. The evaluation months have 169,803 payments. Use the mean cost per call and the median time per call from `usage` to estimate the cost in US dollars (`detector_usd`) and the hours, one call at a time (`detector_hours`). This is an estimate, not a measurement.

In [ ]:
detector_usd = usage["US$"].mean() * 169803
detector_hours = usage["seconds"].median() * 169803 / 3600
print(round(detector_usd, 2), "US$ and", round(detector_hours), "hours, one call at a time")

In [ ]:
expect('the estimated cost in US dollars', detector_usd, 19.65, tolerance=0.01)

In the solution: C-13297 had one amount below the usual one. The estimate for every payment is about US$19.65 and 145 hours one call at a time. The money is small with a small model; the time, the missing labels and the missing customer history are the real problems. And no one has measured whether the model would choose better alerts than the detector.

## Recap

- The model **explains** an alert that the detector found. It does not find the alert. It reads about 20 facts, not the payments table.
- Code checks every summary: numbers and countries must be in the facts, a "First check:" line, no "it is fraud". Test the check with a planted mistake, and do not make it so strict that it cries wolf.
- A number check cannot see a wrong comparison. Keep the facts table next to the summary, so the reviewer can see both.

<img src="https://raw.githubusercontent.com/Nextia-Intelligence/nextia-learning-labs/main/nextia-notebook-branding/assets/section-divider-light.png" alt="" width="100%">

**Next lesson:** [When not to use machine learning](https://learning.nextia-ai.com/courses/anomaly-detection/m06/when-not-to-use-machine-learning/) · **This lesson:** [read it on the site](https://learning.nextia-ai.com/courses/anomaly-detection/m06/where-an-llm-helps/)

<img src="https://raw.githubusercontent.com/Nextia-Intelligence/nextia-learning-labs/main/nextia-notebook-branding/assets/notebook-footer-light.png" alt="Nextia Learning · learning.nextia-ai.com" width="100%">

Free, practical AI courses at [learning.nextia-ai.com](https://learning.nextia-ai.com). The practice data is synthetic and has no real people.